# 🌍 Project 33 — River Width Estimation

## 1. Introduce the Project

### Goal

In this project, we will estimate the width of a river at selected locations.

The main question is:

> How wide is the river at this location?

We will use geometry tools to create a line across the river and measure its length.

For example:

```text
River bank                    River bank
    │                             │
    │<-------- 250 meters ------->│
    │                             │
```

The final output will be:

* A river image
* Measurement lines at selected locations
* Estimated river widths in meters

---

## Why is this useful in Remote Sensing?

River width can change because of:

* Seasonal water changes
* Flooding
* Drought
* Erosion
* Sediment deposition
* Dam operations

By measuring river width at different locations or dates, we can monitor river changes.

Real-world applications include:

* Hydrology
* Flood monitoring
* River morphology
* Environmental monitoring
* Drought analysis

---

## Main New Skills

In this project, you will learn:

* `ee.Geometry.LineString()`
* Creating measurement lines
* `.length()`
* Distance measurements
* Working with geometry objects

---

# 2. Big Picture

```text
Satellite Image
      ↓
Identify River Banks
      ↓
Draw Line Across River
      ↓
Line Geometry
      ↓
Calculate Length
      ↓
Estimated River Width
```

The key idea is simple:

```text
One Line
   ↓
From one river bank
   ↓
To the opposite river bank
   ↓
Line Length = Estimated Width
```

---

# 3. Main Concept — Geometry Distance Measurement

Previously, we mainly used geometries to define a study area.

For example:

```text
Point
Polygon
Study Area
```

Now we use geometry for a different purpose:

> **Measuring distance.**

We create a line:

```text
Bank A ●────────────● Bank B
```

Then ask GEE:

> How long is this line?

The answer can be returned in meters.

---

# 4. Step 1 — Define the Study Area

First, choose a section of a river.

You can draw a polygon around the river in the GEE Code Editor.

```javascript
var studyArea = geometry;
```

Then display it:

```javascript
Map.centerObject(studyArea, 13);
```

Use a relatively high zoom level because river-width estimation needs more detailed visual inspection.

---

# 5. Step 2 — Load a Sentinel-2 Image

We use:

```javascript
"COPERNICUS/S2_SR_HARMONIZED"
```

For example:

```javascript
var image = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(studyArea)
.filterDate("2025-06-01", "2025-07-01")
.filter(ee.Filter.lt(
  "CLOUDY_PIXEL_PERCENTAGE",
  20
))
.median()
.clip(studyArea);
```

The workflow is familiar:

```text
Sentinel-2 ImageCollection
          ↓
Filter Study Area
          ↓
Filter Date
          ↓
median()
          ↓
One Image
```

---

# 6. Step 3 — Display the River

We can display the image in true color:

```javascript
Map.addLayer(
  image,
  {
    bands: ["B4", "B3", "B2"],
    min: 0,
    max: 3000
  },
  "River Image"
);
```

Now zoom in and identify the two river banks.

Conceptually:

```text
Land        Water          Land

███████     ~~~~~~~        ███████
          ↑       ↑
       Bank A   Bank B
```

---

# 7. Step 4 — Create a Measurement Line

Now we create a line from one bank to the other.

Example:

```javascript
var widthLine = ee.Geometry.LineString([
  [longitude1, latitude1],
  [longitude2, latitude2]
]);
```

You would replace the coordinates with your selected points.

The structure is:

```text
Point 1
   ↓
One river bank

Point 2
   ↓
Opposite river bank
```

The line connects them:

```text
Point 1 ●────────────● Point 2
```

---

## Important: Draw the Line Across the River

The line should cross the river approximately perpendicular to the river direction.

Better:

```text
River direction
     ↓

~~~~~~~~~~~~~~
      │
      │
      │ Measurement
      │
~~~~~~~~~~~~~~
```

Not:

```text
~~~~~~~~~~~~~~
   ────────
~~~~~~~~~~~~~~
```

A diagonal or river-parallel line can overestimate the width.

---

# 8. Step 5 — Calculate the Length

Now we use:

```javascript
var widthMeters = widthLine.length();
```

## What does `.length()` do?

It measures the length of the geometry.

```text
LineString
    ↓
.length()
    ↓
Distance
```

The result is approximately the line length in meters.

We can print it:

```javascript
print(
  "Estimated River Width (meters)",
  widthMeters
);
```

---

# 9. Step 6 — Display the Measurement Line

```javascript
Map.addLayer(
  widthLine,
  {
    color: "red"
  },
  "River Width"
);
```

Now you can visually check:

```text
Satellite Image
      +
Measurement Line
```

This is important.

Always inspect whether your line really goes:

```text
Bank
 ↓
Water
 ↓
Opposite Bank
```

---

# 10. Measuring Multiple Locations

One river may have different widths.

For example:

```text
Location A → 100 m

Location B → 180 m

Location C → 90 m
```

You can create multiple lines.

```javascript
var line1 = ee.Geometry.LineString([
  [longitude1, latitude1],
  [longitude2, latitude2]
]);

var line2 = ee.Geometry.LineString([
  [longitude3, latitude3],
  [longitude4, latitude4]
]);

var line3 = ee.Geometry.LineString([
  [longitude5, latitude5],
  [longitude6, latitude6]
]);
```

Then measure each:

```javascript
print("Width 1", line1.length());

print("Width 2", line2.length());

print("Width 3", line3.length());
```

Conceptually:

```text
River
══════════════════════════

       │
       │  Location 1
       │

              │
              │  Location 2
              │

    │
    │  Location 3
```

---

# 11. Complete Workflow

```text
Choose River
      ↓
Select Satellite Image
      ↓
Identify River Banks
      ↓
Create Line Across River
      ↓
ee.Geometry.LineString()
      ↓
.length()
      ↓
River Width in Meters
```

---

# Complete Code

```javascript
// ========================================
// 1. Study Area
// ========================================

var studyArea = geometry;


// ========================================
// 2. Load Sentinel-2 Image
// ========================================

var image = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(studyArea)
.filterDate(
  "2025-06-01",
  "2025-07-01"
)
.filter(
  ee.Filter.lt(
    "CLOUDY_PIXEL_PERCENTAGE",
    20
  )
)
.median()
.clip(studyArea);


// ========================================
// 3. Display Satellite Image
// ========================================

Map.centerObject(
  studyArea,
  13
);

Map.addLayer(
  image,
  {
    bands: ["B4", "B3", "B2"],
    min: 0,
    max: 3000
  },
  "River Image"
);


// ========================================
// 4. Create Width Measurement Line
// ========================================

// Replace these coordinates with
// points on opposite river banks

var widthLine = ee.Geometry.LineString([
  
  [longitude1, latitude1],
  
  [longitude2, latitude2]
  
]);


// ========================================
// 5. Calculate River Width
// ========================================

var widthMeters =
  widthLine.length();

print(
  "Estimated River Width (meters)",
  widthMeters
);


// ========================================
// 6. Display Measurement Line
// ========================================

Map.addLayer(
  widthLine,
  {
    color: "red"
  },
  "River Width"
);
```

---

# Things to Remember

## 1. A `LineString` is a geometry

Previously, we used:

```text
Point
Polygon
```

Now:

```text
LineString
```

A `LineString` connects points with a line.

---

## 2. `.length()` measures the line

```javascript
widthLine.length()
```

means:

```text
This geometry
      ↓
Measure its length
      ↓
Distance
```

For our project:

```text
Line length ≈ River width
```

---

## 3. This is an estimate

The result depends on:

* Image resolution
* How accurately you identify the river bank
* Where you place the measurement line
* River shape

For Sentinel-2, the spatial resolution is 10 m for the visible and NIR bands used here.

Therefore, this simple visual method is more suitable for **relatively wide rivers** and educational analysis.

---

## 4. The measurement line should be approximately perpendicular

Correct:

```text
River direction →

    │
    │ Width
    │
```

Less accurate:

```text
River direction →

       ╲
        ╲ Width
         ╲
```

A diagonal line is longer than the true cross-river width.

---

### 5. A Simple Method to Display Coordinates by Clicking on the Map

Add this code to your GEE script:

```javascript
Map.onClick(function(coords) {
  print("Longitude:", coords.lon);
  print("Latitude:", coords.lat);
});

---

# Common Beginner Mistakes

## ❌ Mistake 1 — Measuring along the river

Incorrect:

```text
River direction →

────────────────
```

This measures river length, not width.

Correct:

```text
River direction →

       │
       │
       │
```

Measure across the river.

---

## ❌ Mistake 2 — Using a line that starts outside the river bank

For example:

```text
Land      River      Land

●───────────────●
```

If your points are far inside the surrounding land, the result will be too large.

Choose points close to the visible water boundaries.

---

## ❌ Mistake 3 — Measuring a narrow river with Sentinel-2

If a river is only a few meters wide:

```text
River width < pixel size
```

Sentinel-2 may not provide enough detail for reliable width estimation.

Remember:

> The satellite resolution limits the detail you can measure.

---

## ❌ Mistake 4 — Treating one measurement as the width of the entire river

A river may be:

```text
Narrow
  ↓
Wide
  ↓
Narrow
  ↓
Wide
```

One measurement only represents that specific location.

For research, measure multiple cross-sections.

---

## ❌ Mistake 5 — Ignoring the observation date

River width can change over time.

For example:

```text
Dry season → Narrower

Flood season → Wider
```

Always record the date or period of the satellite image.

---

# Your Practice — Project 33.1

## Measure River Width at Three Locations

Choose a river section.

Your task:

### 1. Define a study area

Create a geometry covering the river section.

### 2. Load Sentinel-2 imagery

Use:

```javascript
"COPERNICUS/S2_SR_HARMONIZED"
```

Choose a suitable date range and create a composite.

### 3. Create three measurement lines

```text
River

Location 1
     │

Location 2
     │

Location 3
     │
```

Each line should cross the river from one bank to the other.

### 4. Calculate the width

Use:

```javascript
.length()
```

for each line.

### 5. Print your results

Your output should look conceptually like:

```text
Location 1 → ___ meters

Location 2 → ___ meters

Location 3 → ___ meters
```

---

# Conceptual Questions

### 1.

Why should the measurement line be approximately perpendicular to the river direction?

---

### 2.

If Sentinel-2 has a spatial resolution of approximately 10 meters, why might measuring a 5-meter-wide river be unreliable?

---

### 3.

If the same river is measured in July and then again in January, why might the measured widths be different?
